# ECG Digitizer - Offline Baseline Submission

This notebook runs the ECG digitizer in offline mode using pre-packaged dependencies.

## Requirements
1. Upload ECG Digitizer source code and weights as a Kaggle Dataset
2. Attach the dataset to this notebook
3. Run all cells to generate `submission.csv`

## Dataset Structure Expected:
```
/kaggle/input/ecg-digitizer/
├── src/
│   ├── model/
│   ├── config/
│   └── kaggle_inference.py
└── weights/
    ├── unet_best.pth
    └── leadnet_best.pth
```

## Cell 1: Setup Environment

In [ ]:
import sys
import os
from pathlib import Path
import shutil

# Setup paths
DATASET_PATH = Path('/kaggle/input/ecg-digitizer')
WORKING_DIR = Path('/kaggle/working')

print("Setting up environment...")

# Copy source code to working directory
if (DATASET_PATH / 'src').exists():
    shutil.copytree(DATASET_PATH / 'src', WORKING_DIR / 'src', dirs_exist_ok=True)
    print(f"✅ Copied source code from dataset")
else:
    print("❌ Dataset not found! Make sure you've attached the ecg-digitizer dataset")
    print("   Go to 'Add Data' → Search for your uploaded dataset → Add")
    sys.exit(1)

# Copy weights to working directory
if (DATASET_PATH / 'weights').exists():
    shutil.copytree(DATASET_PATH / 'weights', WORKING_DIR / 'weights', dirs_exist_ok=True)
    print(f"✅ Copied model weights from dataset")
else:
    print("❌ Weights not found in dataset!")
    sys.exit(1)

# Add to Python path
sys.path.insert(0, str(WORKING_DIR))

print("\n✅ Environment setup complete!")

## Cell 2: Verify Dependencies

In [ ]:
# Verify required packages (should be available in Kaggle by default)
import numpy as np
import scipy
import torch
import cv2
import pandas as pd
import matplotlib
from sklearn import __version__ as sklearn_version

print("Package versions:")
print(f"  NumPy: {np.__version__}")
print(f"  SciPy: {scipy.__version__}")
print(f"  PyTorch: {torch.__version__}")
print(f"  OpenCV: {cv2.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  Matplotlib: {matplotlib.__version__}")
print(f"  Scikit-learn: {sklearn_version}")

# Check CUDA availability
if torch.cuda.is_available():
    print(f"\n✅ CUDA available: {torch.cuda.get_device_name(0)}")
else:
    print("\n⚠️ CUDA not available, using CPU (slower but will work)")

print("\n✅ All dependencies verified!")

## Cell 3: Configure Inference (Baseline Mode)

In [ ]:
from yacs.config import CfgNode as CN

# Create baseline configuration
# This uses ONLY the proven baseline approach (no TTA, no constraints)
config = CN()

# Model paths
config.UNET_WEIGHTS = str(WORKING_DIR / 'weights' / 'unet_best.pth')
config.LEADNET_WEIGHTS = str(WORKING_DIR / 'weights' / 'leadnet_best.pth')

# Inference settings
config.INFERENCE = CN()
config.INFERENCE.device = 'cuda' if torch.cuda.is_available() else 'cpu'
config.INFERENCE.batch_size = 1
config.INFERENCE.fs = 500  # Sampling frequency (Hz)
config.INFERENCE.duration_lead_ii = 10.0  # Lead II: 10 seconds
config.INFERENCE.duration_others = 2.5  # Other leads: 2.5 seconds
config.INFERENCE.submission_path = str(WORKING_DIR / 'submission.csv')

# Strategy settings (BASELINE = all disabled for reliability)
config.STRATEGIES = CN()
config.STRATEGIES.use_tta = False  # No TTA for baseline
config.STRATEGIES.tta_n_augmentations = 0
config.STRATEGIES.use_physiological_constraints = False  # No constraints for baseline
config.STRATEGIES.constraint_alpha = 0.0

# Data paths (Kaggle competition input)
config.DATA = CN()
config.DATA.test_csv = '/kaggle/input/liverpool-ion-switching/test.csv'  # Replace with actual competition path
config.DATA.test_images_dir = '/kaggle/input/liverpool-ion-switching/test/'  # Replace with actual competition path

print("Configuration created:")
print(f"  Device: {config.INFERENCE.device}")
print(f"  TTA: {config.STRATEGIES.use_tta}")
print(f"  Constraints: {config.STRATEGIES.use_physiological_constraints}")
print(f"  Output: {config.INFERENCE.submission_path}")
print("\n✅ Baseline configuration ready!")

## Cell 4: Load Models and Run Inference

**IMPORTANT:** Update the test data paths in Cell 3 before running this cell!
- `config.DATA.test_csv` should point to the competition's `test.csv`
- `config.DATA.test_images_dir` should point to the test images directory

In [ ]:
from src.kaggle_inference import main

print("Starting ECG digitizer inference...")
print("="*60)

try:
    # Run inference
    main(config)
    
    print("="*60)
    print("✅ Inference completed successfully!")
    
    # Verify output
    if Path(config.INFERENCE.submission_path).exists():
        import pandas as pd
        df = pd.read_csv(config.INFERENCE.submission_path)
        print(f"\nSubmission file created: {config.INFERENCE.submission_path}")
        print(f"  Rows: {len(df):,}")
        print(f"  Columns: {list(df.columns)}")
        print(f"  Value range: [{df['value'].min():.6f}, {df['value'].max():.6f}]")
        print(f"  NaN count: {df['value'].isna().sum()}")
        
        if df['value'].isna().any():
            print("\n⚠️ WARNING: Submission contains NaN values!")
        else:
            print("\n✅ Submission looks good! Ready to submit.")
    else:
        print(f"\n❌ ERROR: Submission file not created at {config.INFERENCE.submission_path}")
        
except Exception as e:
    print(f"\n❌ ERROR during inference: {e}")
    import traceback
    traceback.print_exc()
    raise

## Cell 5: Validate Submission Format

In [ ]:
import pandas as pd

# Load and validate submission
submission = pd.read_csv(config.INFERENCE.submission_path)

print("Submission validation:")
print(f"  Shape: {submission.shape}")
print(f"  Columns: {list(submission.columns)}")

# Check required columns
required_cols = ['id', 'value']
if all(col in submission.columns for col in required_cols):
    print(f"  ✅ Required columns present: {required_cols}")
else:
    print(f"  ❌ Missing required columns!")

# Check for NaN values
nan_count = submission['value'].isna().sum()
if nan_count == 0:
    print(f"  ✅ No NaN values")
else:
    print(f"  ❌ {nan_count} NaN values found ({nan_count/len(submission)*100:.2f}%)")

# Check ID format (should be: image_id_row_lead)
sample_ids = submission['id'].head(5)
print(f"\nSample IDs:")
for id_val in sample_ids:
    print(f"  {id_val}")

# Statistical summary
print(f"\nValue statistics:")
print(submission['value'].describe())

print("\n" + "="*60)
if nan_count == 0:
    print("✅ Submission is valid and ready to submit!")
    print(f"   File: {config.INFERENCE.submission_path}")
else:
    print("⚠️ Please fix NaN values before submitting")

## Next Steps

1. **Review the validation output above** - Ensure no NaN values and reasonable value ranges
2. **Download submission.csv** from `/kaggle/working/submission.csv`
3. **Submit to competition** via the Kaggle submission interface

---

## Troubleshooting

### Dataset not found
- Make sure you've uploaded the ECG Digitizer files as a Kaggle Dataset
- Attach the dataset to this notebook: Add Data → Your Datasets → ecg-digitizer

### Wrong test data paths
- Update `config.DATA.test_csv` and `config.DATA.test_images_dir` in Cell 3
- These should point to the actual competition test data

### CUDA out of memory
- This shouldn't happen with batch_size=1, but if it does:
- Change `config.INFERENCE.device = 'cpu'` in Cell 3

### NaN values in output
- This usually means lead detection failed for some images
- Check the log output for specific errors
- Consider adjusting detection thresholds in the source code